In [18]:
import tslab.metrics as M
print("tslab.metrics contents:")
for name in dir(M):
    if not name.startswith("_"):
        obj = getattr(M, name)
        if callable(obj):
            try:
                sig = str(__import__("inspect").signature(obj))
            except Exception:
                sig = "(sig unavailable)"
            print(f"  {name}{sig}")

tslab.metrics contents:
  backtest_summary(daily_returns, periods_per_year=252)
  compare_models(dataset, predictions, naive_name='Naive (no change)')
  diebold_mariano(actual, forecast_a, forecast_b, horizon=1)
  directional_accuracy(actual_return, predicted_return)
  directional_accuracy_test(actual_return, predicted_return)
  dm_verdict(statistic, p_value, level=0.05)
  evaluate_forecast(close_today, close_next_actual, predicted_return, name='model')
  mae(actual, predicted)
  mape(actual, predicted)
  out_of_sample_r2(actual_return, predicted_return)
  rmse(actual, predicted)
  simple_backtest(actual_return, predicted_return, cost_bps=0.0, long_only=False)


In [19]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from tslab.data import load_market_data
from tslab.features import ALL_FEATURES, make_dataset

frame = load_market_data()
data = make_dataset(frame, horizon=1).dropna(subset=["target_return"]).sort_index()

print(f"Rows: {len(data)}")
print(f"Date range: {data.index.min().date()} -> {data.index.max().date()}")
print(f"Feature columns: {len(ALL_FEATURES)}")
print(f"First 5 features: {ALL_FEATURES[:5]}")
print(f"Target stats: mean={data['target_return'].mean():.6f}, std={data['target_return'].std():.6f}")

Loaded 2978 trading days of MSFT, 2015-01-02 to 2026-11-04, with 7 predictor series.
Rows: 2778
Date range: 2015-10-16 -> 2026-11-03
Feature columns: 47
First 5 features: ['ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5']
Target stats: mean=0.000938, std=0.017332


In [14]:
X         = data[ALL_FEATURES].values
y         = data["target_return"].values
price_t   = data["close"].values
price_t1  = data["target_close"].values
dates     = data.index

INITIAL_TRAIN = 756   # ~3 years
print(f"Total observations: {len(y)}")
print(f"Initial train: {INITIAL_TRAIN}, test steps: {len(y) - INITIAL_TRAIN}")

model_pred_returns = []
naive_pred_returns = []

for t in range(INITIAL_TRAIN, len(y)):
    X_train, y_train = X[:t], y[:t]
    X_test = X[t:t+1]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)   # FIT ONLY ON TRAINING BLOCK
    X_test_s  = scaler.transform(X_test)        # transform with training stats

    model = Ridge(alpha=1.0)
    model.fit(X_train_s, y_train)

    model_pred_returns.append(model.predict(X_test_s)[0])
    naive_pred_returns.append(0.0)              # naive return = 0

model_pred_returns = np.array(model_pred_returns)
naive_pred_returns = np.array(naive_pred_returns)

# Slice aligned arrays
close_t      = price_t[INITIAL_TRAIN:]
close_t1     = price_t1[INITIAL_TRAIN:]
test_dates   = dates[INITIAL_TRAIN:]

model_prices = close_t * np.exp(model_pred_returns)
naive_prices = close_t * np.exp(naive_pred_returns)

print(f"Walk-forward steps completed: {len(model_prices)}")
print(f"First test date: {test_dates[0].date()}, last test date: {test_dates[-1].date()}")

Total observations: 2778
Initial train: 756, test steps: 2022
Walk-forward steps completed: 2022
First test date: 2018-10-17, last test date: 2026-11-03


In [15]:
def compute_metrics(actual, pred, name):
    err = actual - pred
    return {
        "Model": name,
        "RMSE": float(np.sqrt(np.mean(err**2))),
        "MAE":  float(np.mean(np.abs(err))),
        "OOS_R2": float(1 - np.sum(err**2) / np.sum((actual - actual.mean())**2)),
        "DirAcc": float(np.mean(np.sign(np.diff(actual)) == np.sign(np.diff(pred)))),
    }

results = pd.DataFrame([
    compute_metrics(close_t1, model_prices, "Ridge"),
    compute_metrics(close_t1, naive_prices, "Naive"),
]).set_index("Model")

print(results.round(6))

           RMSE       MAE    OOS_R2    DirAcc
Model                                        
Ridge  5.580020  3.784501  0.997917  0.485403
Naive  5.427925  3.651952  0.998029  0.488372


In [16]:
# Try the repo's function first; fall back to a manual implementation
try:
    from tslab.metrics import diebold_mariano, dm_verdict
    dm_stat, p_value = diebold_mariano(close_t1, model_prices, naive_prices, horizon=1)
    verdict = dm_verdict(dm_stat, p_value)
    source = "tslab.metrics"
except Exception as e:
    print(f"tslab.metrics.diebold_mariano failed ({type(e).__name__}: {e}). Using manual DM.")
    from scipy.stats import norm
    d = (close_t1 - model_prices)**2 - (close_t1 - naive_prices)**2
    n = len(d)
    mean_d = d.mean()
    var_d  = np.var(d, ddof=0) / n
    dm_stat = mean_d / np.sqrt(var_d)
    p_value = float(2 * (1 - norm.cdf(abs(dm_stat))))
    # Sign convention: negative statistic -> first forecast (Ridge) better
    if p_value >= 0.05:
        verdict = "No significant difference"
    elif dm_stat < 0:
        verdict = "Significantly BETTER than naive"
    else:
        verdict = "Significantly WORSE than naive"
    source = "manual fallback"

print(f"DM statistic: {dm_stat:.4f}")
print(f"DM p-value:   {p_value:.4f}")
print(f"Verdict:      {verdict}")
print(f"(Source: {source})")

DM statistic: 3.3353
DM p-value:   0.0009
Verdict:      Significantly WORSE than naive
(Source: tslab.metrics)


In [17]:
wf_output = pd.DataFrame({
    "date": test_dates,
    "close_t": close_t,
    "actual_next_close": close_t1,
    "ridge_pred_return": model_pred_returns,
    "ridge_price": model_prices,
    "naive_price": naive_prices,
})
wf_output.to_csv("./data/walkforward_ridge_msft.csv", index=False)
print(f"Saved {len(wf_output)} rows to ./data/walkforward_ridge_msft.csv")
print(wf_output.head())

Saved 2022 rows to ./data/walkforward_ridge_msft.csv
        date     close_t  actual_next_close  ridge_pred_return  ridge_price  \
0 2018-10-17  102.292061         100.250092           0.008115   103.125497   
1 2018-10-18  100.250092         100.397957           0.010600   101.318433   
2 2018-10-19  100.397957         101.294174           0.014436   101.857822   
3 2018-10-22  101.294174          99.880531           0.013856   102.707515   
4 2018-10-23   99.880531          94.540009           0.014901   101.379945   

   naive_price  
0   102.292061  
1   100.250092  
2   100.397957  
3   101.294174  
4    99.880531  


In [8]:
y_test = y[INITIAL_TRAIN:]

print(f"Mean ridge predicted return: {model_pred_returns.mean():.6f}")
print(f"Mean actual return:          {y_test.mean():.6f}")
print(f"Std of ridge predictions:    {model_pred_returns.std():.6f}")
print(f"Std of actual returns:       {y_test.std():.6f}")
print()

rmse_ridge_ret = np.sqrt(np.mean((model_pred_returns - y_test)**2))
rmse_naive_ret = np.sqrt(np.mean(y_test**2))
print(f"Return RMSE (Ridge):     {rmse_ridge_ret:.6f}")
print(f"Return RMSE (Naive=0):   {rmse_naive_ret:.6f}")
print()

r2_ridge = 1 - np.sum((model_pred_returns - y_test)**2) / np.sum((y_test - y_test.mean())**2)
r2_naive = 1 - np.sum((0 - y_test)**2) / np.sum((y_test - y_test.mean())**2)
print(f"Return R² (Ridge): {r2_ridge:.6f}")
print(f"Return R² (Naive): {r2_naive:.6f}")

Mean ridge predicted return: 0.000721
Mean actual return:          0.000836
Std of ridge predictions:    0.005820
Std of actual returns:       0.018552

Return RMSE (Ridge):     0.018884
Return RMSE (Naive=0):   0.018571

Return R² (Ridge): -0.036023
Return R² (Naive): -0.002031


In [11]:
def wf_ridge(X, y, initial_train, alpha=1.0, shrink=1.0, fit_intercept=True):
    preds = []
    for t in range(initial_train, len(y)):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[:t])
        Xte = scaler.transform(X[t:t+1])
        m = Ridge(alpha=alpha, fit_intercept=fit_intercept)
        m.fit(Xtr, y[:t])
        preds.append(m.predict(Xte)[0] * shrink)
    return np.array(preds)

shrink_preds  = wf_ridge(X, y, INITIAL_TRAIN, alpha=1.0, shrink=0.05)
no_int_preds  = wf_ridge(X, y, INITIAL_TRAIN, alpha=1.0, fit_intercept=False)
heavy_preds   = wf_ridge(X, y, INITIAL_TRAIN, alpha=1000.0)

def rmse(a, p): return np.sqrt(np.mean((a-p)**2))
print(f"Original Ridge:       RMSE {rmse(close_t1, model_prices):.4f}")
print(f"Shrink 0.05:          RMSE {rmse(close_t1, close_t * np.exp(shrink_preds)):.4f}")
print(f"No intercept:         RMSE {rmse(close_t1, close_t * np.exp(no_int_preds)):.4f}")
print(f"Heavy Ridge (α=1000): RMSE {rmse(close_t1, close_t * np.exp(heavy_preds)):.4f}")
print(f"Naive:                RMSE {rmse(close_t1, naive_prices):.4f}")

Original Ridge:       RMSE 5.5800
Shrink 0.05:          RMSE 5.4250
No intercept:         RMSE 5.5891
Heavy Ridge (α=1000): RMSE 5.4937
Naive:                RMSE 5.4279


In [20]:
from tslab.metrics import diebold_mariano, dm_verdict

shrink_prices = close_t * np.exp(shrink_preds)
dm_stat_shrink, p_shrink = diebold_mariano(close_t1, shrink_prices, naive_prices, horizon=1)

print(f"Shrink 0.05 vs Naive:")
print(f"  DM statistic: {dm_stat_shrink:.4f}")
print(f"  DM p-value:   {p_shrink:.4f}")
print(f"  Verdict:      {dm_verdict(dm_stat_shrink, p_shrink)}")

Shrink 0.05 vs Naive:
  DM statistic: -1.2726
  DM p-value:   0.2033
  Verdict:      No significant difference


In [21]:
train_means = np.array([y[:t].mean() for t in range(INITIAL_TRAIN, len(y))])
mean_prices = close_t * np.exp(train_means)

print(f"Constant-mean RMSE: {rmse(close_t1, mean_prices):.4f}")
print(f"Naive RMSE:         {rmse(close_t1, naive_prices):.4f}")

dm_stat_mean, p_mean = diebold_mariano(close_t1, mean_prices, naive_prices, horizon=1)
print(f"DM (mean vs naive): stat {dm_stat_mean:.4f}, p {p_mean:.4f}")
print(f"Verdict: {dm_verdict(dm_stat_mean, p_mean)}")

Constant-mean RMSE: 5.4284
Naive RMSE:         5.4279
DM (mean vs naive): stat 0.0548, p 0.9563
Verdict: No significant difference


In [22]:
from tslab.data import load_market_data
from tslab.features import ALL_FEATURES, make_dataset
import numpy as np, pandas as pd

frame = load_market_data()
data_all = make_dataset(frame, horizon=1)                     # keep last row (target may be NaN)
data_train = data_all.dropna(subset=["target_return"]).sort_index()

y_all = data_train["target_return"].values
mu    = y_all.mean()                                          # daily drift
sigma = y_all.std()                                           # daily vol

last_date  = data_train.index[-1]
last_price = data_train["close"].iloc[-1]

horizons   = np.arange(1, 64)
dates_fc   = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=63)
z90        = 1.2816                                           # 10-90 band

forecast = last_price * np.exp(mu * horizons)
lower    = last_price * np.exp(mu * horizons - z90 * sigma * np.sqrt(horizons))
upper    = last_price * np.exp(mu * horizons + z90 * sigma * np.sqrt(horizons))

forecast_df = pd.DataFrame({
    "date":     dates_fc,
    "forecast": forecast,
    "lower":    lower,
    "upper":    upper,
})
forecast_df.to_csv("./data/forecast_basic.csv", index=False)
print(forecast_df.head())
print(forecast_df.tail())
print(f"Saved {len(forecast_df)} rows to ./data/forecast_basic.csv")

Loaded 2978 trading days of MSFT, 2015-01-02 to 2026-11-04, with 7 predictor series.
        date    forecast       lower       upper
0 2026-11-04  520.473431  509.042129  532.161440
1 2026-11-05  520.961786  504.854239  537.583249
2 2026-11-06  521.450598  501.773615  541.899212
3 2026-11-09  521.939869  499.264627  545.644960
4 2026-11-10  522.429600  497.120030  549.027740
         date    forecast       lower       upper
58 2027-01-25  549.568857  463.381519  651.786737
59 2027-01-26  550.084511  463.149099  653.338136
60 2027-01-27  550.600649  462.922327  654.885403
61 2027-01-28  551.117271  462.701058  656.428685
62 2027-01-29  551.634379  462.485155  657.968119
Saved 63 rows to ./data/forecast_basic.csv


In [4]:
import numpy as np
import pandas as pd
from tslab.data import load_market_data
from tslab.features import make_dataset

print("Loading data and generating forecasts...")

# 1. Reload the data from scratch
frame = load_market_data()
data = make_dataset(frame, horizon=1).dropna(subset=["target_return"]).sort_index()

y = data["target_return"].values
price_t = data["close"].values
dates = data.index

# 2. Calculate forecast parameters (Drift model)
mu = y.mean()
sigma = y.std()
last_date = dates[-1]
last_price = price_t[-1]

# 3. Generate 63-day forecast with bounds
horizons = np.arange(1, 64)
dates_fc = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=63)
z90 = 1.2816  # 10-90% band

forecast = last_price * np.exp(mu * horizons)
lower = last_price * np.exp(mu * horizons - z90 * sigma * np.sqrt(horizons))
upper = last_price * np.exp(mu * horizons + z90 * sigma * np.sqrt(horizons))

forecast_df = pd.DataFrame({
    "date": dates_fc,
    "forecast": forecast,
    "lower": lower,
    "upper": upper,
})
forecast_df.to_csv("./data/forecast_basic.csv", index=False)
print(f"\n✅ Saved 63 rows to ./data/forecast_basic.csv")
print(forecast_df.head(3))

# 4. Save actual last year for Power BI
cutoff = last_date - pd.Timedelta(days=365)
mask = data.index >= cutoff
actual_last_year = data.loc[mask, ["close"]].reset_index()
actual_last_year.columns = ["date", "close"]
actual_last_year.to_csv("./data/actual_last_year.csv", index=False)

print(f"\n✅ Saved {len(actual_last_year)} rows to ./data/actual_last_year.csv")
print(actual_last_year.head(3))

Loading data and generating forecasts...
Loaded 2978 trading days of MSFT, 2015-01-02 to 2026-11-04, with 7 predictor series.

✅ Saved 63 rows to ./data/forecast_basic.csv
        date    forecast       lower       upper
0 2026-11-04  520.473431  509.042129  532.161440
1 2026-11-05  520.961786  504.854239  537.583249
2 2026-11-06  521.450598  501.773615  541.899212

✅ Saved 252 rows to ./data/actual_last_year.csv
        date       close
0 2025-11-03  511.822021
1 2025-11-04  509.149231
2 2025-11-05  502.051453


In [5]:
import pandas as pd

# 1. Load both files
actuals = pd.read_csv("./data/actual_last_year.csv")
forecasts = pd.read_csv("./data/forecast_basic.csv")

# 2. Add empty columns to match the shapes
actuals["forecast"] = None
actuals["lower"] = None
actuals["upper"] = None

forecasts["close"] = None

# 3. Combine them into one timeline
combined = pd.concat([actuals, forecasts])
combined["date"] = pd.to_datetime(combined["date"])
combined = combined.sort_values("date")

# 4. Save as a single file for Power BI
combined.to_csv("./data/powerbi_master.csv", index=False)
print("✅ Saved combined file to ./data/powerbi_master.csv")

✅ Saved combined file to ./data/powerbi_master.csv
